# 01 — Região de saúde atribuída a cada internação

**Story:** US-02 — **O que este notebook faz:** pega a base tratada de internações
(`sih_pb_2025_tratado.parquet`) e adiciona duas colunas novas: `regiao_res` (a região
de saúde onde o paciente MORA) e `regiao_int` (a região de saúde onde ele foi
INTERNADO). Quando as duas são diferentes, o paciente viajou para se tratar — é
exatamente esse movimento que o projeto quer medir.

**Por que região de saúde e não município?** A Paraíba tem 223 municípios, mas o SUS
organiza o atendimento em **16 regiões de saúde** — grupos oficiais de municípios que
deveriam se bastar na maior parte dos atendimentos. A Secretaria de Saúde planeja por
região, então nossa matriz origem→destino precisa falar a língua dela.

## 1. A fonte oficial: base territorial do DATASUS

A pergunta "qual município pertence a qual região de saúde?" tem uma resposta oficial:
a **base territorial do DATASUS** — o mesmo conjunto de tabelas que o TabWin (o
programa oficial de tabulação do Ministério da Saúde) usa. Não é uma lista montada à
mão por terceiros: é o próprio Ministério dizendo quem pertence a quê.

**Registro do download (reprodutibilidade):**
- Servidor: `ftp.datasus.gov.br` (FTP público, mesmo servidor de onde baixamos o SIH)
- Arquivo original: `/territorio/tabelas/2025/10-base_territorial_out25.zip`
- Data do download: **23/07/2026**
- Cópia local congelada: `data/raw/base_territorial_out25.zip` (fora do git, como todo bruto)

Escolhemos a versão de **outubro/2025** por ser a última publicada em 2025 — o mesmo
ano das internações que analisamos (as versões de 2026 existem, mas manter tudo no
mesmo ano evita qualquer mudança de recorte no meio do caminho).

Dentro do zip usamos três arquivos, todos da mesma fonte:
- `rl_municip_regsaud.csv` — o vínculo município → código da região de saúde
- `tb_municip.csv` — o nome de cada município
- `tb_regsaud.csv` — o nome de cada região de saúde

O download + montagem + validação viram código reaproveitável em
`src/baixar_base_territorial.py` (mesmo padrão do `src/congelar_sih.py`: se o arquivo
já existe, não baixa de novo). O notebook chama esse código e mostra os resultados.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

# O script em src/ usa caminhos relativos à raiz do projeto (data/raw, data/processed),
# então mudamos o diretório de trabalho para a raiz antes de usá-lo.
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

import baixar_base_territorial as bt

print(f"pandas {pd.__version__}")
print(f"Trabalhando em: {Path.cwd().name}")

pandas 2.3.3
Trabalhando em: projeto-saude


## 2. Baixar (se preciso) e montar a tabela município → região de saúde da PB

Três passos, todos dentro do script:
1. **Baixar o zip** do FTP do DATASUS — pulado se `data/raw/base_territorial_out25.zip` já existe.
2. **Filtrar a PB**: todo código de município da Paraíba começa com `25` (é o código do estado no IBGE).
3. **Juntar os nomes**: o arquivo de vínculo só tem códigos, então buscamos o nome do
   município e o nome da região nas duas tabelas auxiliares do MESMO zip — nunca
   misturamos fontes diferentes.

> **Surpresa encontrada no caminho:** os CSVs do DATASUS não seguem um padrão único nem
> dentro do mesmo zip — `tb_municip.csv` separa colunas com `;` e os outros dois com `,`.
> Por isso o leitor no script detecta o separador automaticamente em vez de fixar um.

In [2]:
bt.baixar_zip()
regioes_pb = bt.montar_tabela_pb()
regioes_pb.head(8)

zip já baixado: base_territorial_out25.zip (1,699,756 bytes)


,codigo_municipio,nome_municipio,codigo_regiao_saude,nome_regiao_saude
0,250010,Agua Branca,25011,11ª Região - PB
1,250020,Aguiar,25007,7ª Região - PB
2,250030,Alagoa Grande,25003,3ª Região - PB
3,250040,Alagoa Nova,25003,3ª Região - PB
4,250050,Alagoinha,25002,2ª Região - PB
5,250053,Alcantil,25015,15ª Região - PB
6,250057,Algodao De Jandaira,25003,3ª Região - PB
7,250060,Alhandra,25001,1ª Região Mata Atlântica - PB


## 3. Validar a tabela antes de usá-la

Antes de aplicar qualquer tabela de referência, conferimos contra fatos que já sabemos
de fora dos dados:
- A PB tem **223 municípios** → a tabela deve ter exatamente 223 linhas.
- Cada município pertence a **exatamente 1** região de saúde → nenhum código repetido.
- A PB tem **16 regiões de saúde** → a coluna de região deve ter 16 valores distintos.

Se qualquer um falhar, a função para tudo com erro — melhor quebrar aqui do que
carregar um mapeamento errado para o resto do projeto.

In [3]:
bt.validar(regioes_pb)

# Quantos municípios tem cada região? (só para conhecer o mapa)
regioes_pb.groupby("nome_regiao_saude", as_index=False).agg(
    n_municipios=("codigo_municipio", "count")
).sort_values("n_municipios", ascending=False)

validação OK: 223 municípios da PB, cada um em exatamente 1 das 16 regiões de saúde


,nome_regiao_saude,n_municipios
8,2ª Região - PB,25
12,6ª Região - PB,24
13,7ª Região - PB,18
11,5ª Região - PB,17
6,16ª Região - PB,15
15,9ª Região - PB,15
5,15ª Região - PB,14
2,12ª Região - PB,14
7,1ª Região Mata Atlântica - PB,14
9,3ª Região - PB,12


## 4. Salvar a tabela de referência (versionada no git)

Salvamos como CSV pequeno em `data/processed/regioes_saude_pb.csv`. Diferente dos
arquivos brutos, este ENTRA no git: quem clonar o repositório (Pedro!) já recebe a
tabela pronta, sem depender do FTP do DATASUS estar no ar.

In [4]:
TABELA = Path("data/processed/regioes_saude_pb.csv")
regioes_pb.to_csv(TABELA, index=False, encoding="utf-8")
print(f"salvo: {TABELA} ({len(regioes_pb)} linhas)")

salvo: data\processed\regioes_saude_pb.csv (223 linhas)


## 5. Aplicar à base de internações

Carregamos a base tratada (US-01) e criamos as duas colunas novas usando o mapeamento
código do município → nome da região:

- **`regiao_int`** (onde o paciente foi internado): `MUNIC_MOV` é sempre um município
  da PB — a base só tem hospitais paraibanos — então TODA linha deve receber região.
- **`regiao_res`** (onde o paciente mora): se `MUNIC_RES` começa com `25`, é residente
  da PB e recebe sua região; senão, marcamos **"Fora da PB"** — pacientes de outros
  estados internados aqui (importação de pacientes, o oposto da evasão).

Usamos `.map()` com um dicionário código→região em vez de `merge` porque é a forma
mais simples de "traduzir" uma coluna inteira — e não corre o risco de duplicar ou
perder linhas.

In [5]:
sih = pd.read_parquet("data/processed/sih_pb_2025_tratado.parquet")
linhas_entrada = len(sih)
print(f"base de entrada: {linhas_entrada:,} internações")

mapa_regiao = dict(
    zip(regioes_pb["codigo_municipio"], regioes_pb["nome_regiao_saude"])
)

# Região do hospital: mapeamento direto (todo hospital é da PB)
sih["regiao_int"] = sih["MUNIC_MOV"].map(mapa_regiao)

# Região de residência: PB -> região; fora da PB -> rótulo fixo
reside_na_pb = sih["MUNIC_RES"].str.startswith("25")
sih["regiao_res"] = sih["MUNIC_RES"].map(mapa_regiao)
sih.loc[~reside_na_pb, "regiao_res"] = "Fora da PB"

sih[["MUNIC_RES", "nome_mun_res", "regiao_res", "MUNIC_MOV", "nome_mun_mov", "regiao_int"]].head(8)

base de entrada: 258,125 internações


,MUNIC_RES,nome_mun_res,regiao_res,MUNIC_MOV,nome_mun_mov,regiao_int
0,251080,Patos,6ª Região - PB,251080,Patos,6ª Região - PB
1,250670,Imaculada,11ª Região - PB,251080,Patos,6ª Região - PB
2,250939,Maturéia,6ª Região - PB,251080,Patos,6ª Região - PB
3,251210,Pombal,13ª Região - PB,250750,João Pessoa,1ª Região Mata Atlântica - PB
4,261160,Recife,Fora da PB,250750,João Pessoa,1ª Região Mata Atlântica - PB
5,250750,João Pessoa,1ª Região Mata Atlântica - PB,250750,João Pessoa,1ª Região Mata Atlântica - PB
6,250750,João Pessoa,1ª Região Mata Atlântica - PB,250750,João Pessoa,1ª Região Mata Atlântica - PB
7,250750,João Pessoa,1ª Região Mata Atlântica - PB,250750,João Pessoa,1ª Região Mata Atlântica - PB


## 6. Validações finais (obrigatórias)

Quatro checagens antes de salvar:

1. **Mesmo número de linhas** que a base de entrada — adicionar colunas não pode criar
   nem sumir com internações.
2. **`regiao_int` sem nenhum vazio** — todo hospital é da PB, então 100% mapeado.
3. **`regiao_res` sem vazios injustificados** — residente da PB recebe região, residente
   de fora recebe "Fora da PB"; nenhuma linha pode ficar sem nada.
4. **Sanity check da distribuição** — as regiões de João Pessoa e Campina Grande
   concentram os grandes hospitais do estado; se elas não dominarem o ranking de
   internações, algo deu errado no mapeamento.

In [6]:
# (a) número de linhas preservado
assert len(sih) == linhas_entrada, "o número de linhas mudou!"
print(f"(a) linhas: {len(sih):,} = {linhas_entrada:,} da entrada — OK")

# (b) regiao_int 100% preenchida
nulos_int = sih["regiao_int"].isna().sum()
assert nulos_int == 0, f"{nulos_int} internações sem região do hospital!"
print(f"(b) regiao_int sem nulos: {nulos_int} vazios — OK")

# (c) regiao_res 100% preenchida (PB -> região; fora -> 'Fora da PB')
nulos_res = sih["regiao_res"].isna().sum()
assert nulos_res == 0, f"{nulos_res} internações sem região de residência!"
fora_pb = (sih["regiao_res"] == "Fora da PB").sum()
print(f"(c) regiao_res sem nulos: {nulos_res} vazios — OK "
      f"({fora_pb:,} internações de residentes de outros estados = 'Fora da PB')")

(a) linhas: 258,125 = 258,125 da entrada — OK
(b) regiao_int sem nulos: 0 vazios — OK
(c) regiao_res sem nulos: 0 vazios — OK (1,502 internações de residentes de outros estados = 'Fora da PB')


In [7]:
# (d) distribuição de internações por região do hospital
distribuicao = (
    sih["regiao_int"].value_counts().rename_axis("regiao_int").reset_index(name="internacoes")
)
distribuicao["%"] = (distribuicao["internacoes"] / len(sih) * 100).round(1)
distribuicao

,regiao_int,internacoes,%
0,1ª Região Mata Atlântica - PB,106543,41.3
1,16ª Região - PB,68051,26.4
2,6ª Região - PB,16964,6.6
3,9ª Região - PB,10968,4.2
4,2ª Região - PB,8573,3.3
5,10ª Região - PB,6426,2.5
6,14ª Região - PB,6168,2.4
7,8ª Região - PB,5567,2.2
8,5ª Região - PB,5466,2.1
9,13ª Região - PB,4552,1.8


## 7. Salvar a base final

Tudo validado — salvamos a base com as duas colunas novas em
`data/processed/sih_pb_2025_regioes.parquet`. É ela que alimenta a próxima etapa:
a matriz origem→destino entre regiões de saúde.

In [8]:
DESTINO = Path("data/processed/sih_pb_2025_regioes.parquet")
sih.to_parquet(DESTINO)
print(f"salvo: {DESTINO} ({len(sih):,} linhas, {sih.shape[1]} colunas)")

salvo: data\processed\sih_pb_2025_regioes.parquet (258,125 linhas, 120 colunas)


## Resumo do que foi feito

1. Baixamos a **base territorial oficial do DATASUS** (out/2025) — a mesma que o
   TabWin do Ministério da Saúde usa — e extraímos a relação município → região de saúde.
2. Validamos contra fatos conhecidos: **223 municípios da PB, cada um em exatamente 1
   das 16 regiões de saúde**.
3. Salvamos a tabela de referência versionada: `data/processed/regioes_saude_pb.csv`.
4. Aplicamos à base de internações: `regiao_int` (100% preenchida) e `regiao_res`
   (residentes de outros estados = "Fora da PB"; zero vazios).
5. Sanity check confirmado: João Pessoa e Campina Grande dominam as internações.
6. Base final salva: `data/processed/sih_pb_2025_regioes.parquet`.